# Session 4: Priors, Posterior Uncertainty & Predictive Checking
### General Laboratory & Demonstration Workbook (Vertical Slice 1)
*Course: Bayesian Analysis of Empirical Data (2026)*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/iknyazeva/bayes-cogsci-book/blob/main/notebooks/colab/04_priors_posterior_uncertainty_predictive_checking.ipynb)

---

## 1. Before Class: The Closed-Form Bayesian Vertical Slice
* **Core Philosophy**: Session 4 delivers the first complete, end-to-end Bayesian analysis pipeline:
  $$\text{Prior Elicitation} \longrightarrow \text{Prior Predictive Check} \longrightarrow \text{Conjugate Updating} \longrightarrow \text{Posterior Uncertainty (ETI vs HDI)} \longrightarrow \text{Posterior Predictive Check} \longrightarrow \text{Prior Sensitivity Audit}$$
* **Why the Beta-Binomial Model?** By using an exact, analytically closed conjugate model, we master the fundamental logic of Bayesian probability, precision-weighted information pooling, and predictive validation without the computational overhead of MCMC samplers.
* **Workbook Structure**:
  1. **Demonstration & Worked Example**: Step-by-step code and interactive visualizations.
  2. **💡 Suggested Experiments**: Targeted ideas to test sample size scaling, prior sensitivity, boundary skew, and predictive risk.
  3. **🧪 Student Sandbox**: Editable code blocks ready for your modifications.
* **Runtime**: ~90 minutes. Click **File $\to$ Save a copy in Drive** before running.


## 2. Environment Check & Setup
Run this cell to import required libraries and configure Plotly for Google Colab.


In [1]:
import sys
import os
import numpy as np
import pandas as pd
from scipy import stats, optimize
import plotly.graph_objects as go
from plotly.subplots import make_subplots

IS_COLAB = "google.colab" in sys.modules

if IS_COLAB:
    print("⚡ Running in Google Colab environment.")
    import plotly.io as pio
    pio.renderers.default = "colab"
else:
    print("💻 Running in local environment.")

RANDOM_SEED = 2026
rng = np.random.default_rng(RANDOM_SEED)
print(f"✅ Environment initialized. NumPy random seed set to {RANDOM_SEED}.")


💻 Running in local environment.
✅ Environment initialized. NumPy random seed set to 2026.


## 3. Observable Learning Targets
By completing this workbook, you will be able to:
1. **Elicit Priors on the Observable Scale**: Translate substantive domain knowledge into Beta hyperparameters $\alpha, \beta$ matching expected proportions and prior effective sample size.
2. **Conduct Prior Predictive Checks**: Simulate fake observable data before seeing real evidence to audit whether candidate priors are reasonable or absurd.
3. **Execute Closed-Form Conjugate Updates**: Analytically compute posterior parameters $\alpha_{\text{post}} = \alpha + k$, $\beta_{\text{post}} = \beta + N - k$ and express the posterior mean as a precision-weighted average.
4. **Compute and Contrast Credible Intervals (ETI vs. HDI)**: Calculate 95% Equal-Tailed Intervals and 95% Highest Density Intervals, identifying when and why they diverge.
5. **Generate Posterior Predictive Forecasts**: Simulate future observations $\tilde{y} \sim \text{Binomial}(\tilde{N}, \theta^{(s)})$ and compute tail-risk threshold probabilities.
6. **Perform a Robustness / Prior Sensitivity Audit**: Evaluate how posterior conclusions shift across flat, weakly informative, skeptical, and informative priors.


## 4. Classwork 0: Predict Before Running
> ✍ **WRITE (Prediction Prompt)**:
> 1. Suppose a domain expert says: *"I expect policy support to be roughly 60%, with uncertainty equivalent to having already surveyed 10 people."* Which Beta distribution represents this prior?
> 2. If an empirical study observes $k = 16$ supporters out of $N = 20$ people, what will be the exact posterior mean under this prior? Will it be closer to 0.60 or to 0.80?


---
## 5. Demonstration 1: Prior Elicitation & Prior Predictive Checking
### A. Worked Example: Evaluating Three Candidate Priors on Observable Data
Before looking at our sample ($N = 20$), we define three candidate priors for policy support $\theta$:
1. **Flat / Uniform Prior**: $\operatorname{Beta}(1, 1)$ — asserts all values $\theta \in [0, 1]$ are equally plausible.
2. **Weakly Informative Prior**: $\operatorname{Beta}(3, 3)$ — weakly centered at 0.50 with prior weight $n_0 = 6$.
3. **Substantive Informed Prior**: $\operatorname{Beta}(6, 4)$ — centered at $\mu_0 = 6/10 = 0.60$ with prior weight $n_0 = 10$.

We draw 2,000 parameter samples from each prior and simulate the **Prior Predictive Distribution** of observable survey counts $y_{\text{sim}} \sim \operatorname{Binomial}(N=20, \theta^{(s)})$.


In [2]:
# 1. Candidate Prior Hyperparameters
priors = {
    'Flat Beta(1, 1)': (1, 1),
    'Weakly Informative Beta(3, 3)': (3, 3),
    'Substantive Informed Beta(6, 4)': (6, 4)
}

S_draws = 3000
N_sample = 20

fig_prior_pred = make_subplots(
    rows=1, cols=3,
    subplot_titles=[
        '<b>Flat Beta(1,1)</b><br><span style="font-size:11px;color:#64748b">Uniform across all k</span>',
        '<b>Weak Beta(3,3)</b><br><span style="font-size:11px;color:#64748b">Centered at 10/20</span>',
        '<b>Informed Beta(6,4)</b><br><span style="font-size:11px;color:#64748b">Centered at 12/20</span>'
    ]
)

colors = ['#94a3b8', '#3b82f6', '#10b981']

for i, (name, (a, b)) in enumerate(priors.items(), start=1):
    # Draw theta from prior
    theta_draws = rng.beta(a, b, size=S_draws)
    # Simulate observable counts
    y_sim = rng.binomial(n=N_sample, p=theta_draws)
    
    # Check probabilities
    p_extreme = np.mean((y_sim <= 2) | (y_sim >= 18))
    print(f"{name:32s}: Prior Mean = {a/(a+b):.2f}, Prior N = {a+b:2d}, Implied P(k ≤ 2 or k ≥ 18) = {p_extreme*100:.1f}%")
    
    fig_prior_pred.add_trace(
        go.Histogram(x=y_sim, histnorm='probability', marker_color=colors[i-1], name=name),
        row=1, col=i
    )
    fig_prior_pred.update_xaxes(title_text='Observable k (out of 20)', range=[-0.5, 20.5], row=1, col=i)
    fig_prior_pred.update_yaxes(title_text='Prior Predictive Prob' if i == 1 else '', row=1, col=i)

fig_prior_pred.update_layout(template='plotly_white', height=380, showlegend=False)
fig_prior_pred.show()


Flat Beta(1, 1)                 : Prior Mean = 0.50, Prior N =  2, Implied P(k ≤ 2 or k ≥ 18) = 28.6%
Weakly Informative Beta(3, 3)   : Prior Mean = 0.50, Prior N =  6, Implied P(k ≤ 2 or k ≥ 18) = 6.9%
Substantive Informed Beta(6, 4) : Prior Mean = 0.60, Prior N = 10, Implied P(k ≤ 2 or k ≥ 18) = 6.5%


### 💡 Suggested Experiments for Prior Elicitation
Try running these experiments in the sandbox cell below:

* **Experiment 1A (The Danger of the Flat Prior)**:
  * Inspect the Flat $\operatorname{Beta}(1, 1)$ panel above. Notice that it assigns a **nearly $15\%$ probability** to seeing extreme un-replicated outcomes ($k \le 2$ or $k \ge 18$). In real social survey research, unanimous consensus ($0\%$ or $100\%$) is virtually impossible. A truly neutral scientific prior is often weakly informative rather than strictly flat!
* **Experiment 1B (Skeptical Prior Elicitation)**:
  * Suppose an intervention claims to improve memory, but historical trials suggest failure. Elicit a skeptical prior centered at $25\%$ success with effective prior sample size $n_0 = 12$ (i.e. $\alpha = 3, \beta = 9$). Simulate its prior predictive distribution for $N = 30$ trials.
* **Experiment 1C (Prior Elicitation Formula)**:
  * For any desired prior mean $\mu_0$ and prior sample size $n_0$, verify that $\alpha = \mu_0 n_0$ and $\beta = (1 - \mu_0) n_0$. Test this with $\mu_0 = 0.70, n_0 = 20$.


In [3]:
# 🧪 STUDENT SANDBOX: Prior Elicitation Testbed
# Enter your desired prior mean and prior effective sample size:
user_mu0 = 0.25   # Expected proportion (e.g. 0.25 for skeptical)
user_n0 = 12      # Prior strength in pseudo-observations

# Compute hyperparameters
alpha_sb = user_mu0 * user_n0
beta_sb = (1 - user_mu0) * user_n0

# Simulate prior predictive
theta_sb = rng.beta(alpha_sb, beta_sb, size=2000)
y_sb = rng.binomial(n=20, p=theta_sb)

print(f"Elicited Hyperparameters: Beta({alpha_sb:.2f}, {beta_sb:.2f})")
print(f"Prior Predictive 90% Interval for k out of 20: [{np.percentile(y_sb, 5):.0f}, {np.percentile(y_sb, 95):.0f}]")
print(f"Probability of k ≥ 15: {np.mean(y_sb >= 15)*100:.2f}%")


Elicited Hyperparameters: Beta(3.00, 9.00)
Prior Predictive 90% Interval for k out of 20: [1, 11]
Probability of k ≥ 15: 0.40%


---
## 6. Demonstration 2: Exact Conjugate Updating & Precision Weighting
### A. Worked Example: Updating with Empirical Survey Data
Now we observe our empirical dataset: **$k = 16$ supporters out of $N = 20$ surveyed** (Sample proportion $\hat{\theta} = 16/20 = 0.80$).

Using the Substantive Informed Prior $\operatorname{Beta}(\alpha=6, \beta=4)$:
* **Prior Mean**: $\mu_0 = 6/10 = 0.60$ with prior weight $n_0 = 10$.
* **Data Evidence**: $\hat{\theta}_{\text{MLE}} = 16/20 = 0.80$ with data weight $N = 20$.
* **Posterior Hyperparameters**:
  $$\alpha_{\text{post}} = \alpha + k = 6 + 16 = 22$$
  $$\beta_{\text{post}} = \beta + N - k = 4 + 4 = 8$$
* **Precision-Weighted Posterior Mean**:
  $$\mathbb{E}[\theta \mid k] = \frac{n_0}{n_0 + N} \mu_0 + \frac{N}{n_0 + N} \hat{\theta}_{\text{MLE}} = \frac{10}{30}(0.60) + \frac{20}{30}(0.80) = 0.20 + 0.5333 = \mathbf{0.7333}$$


In [4]:
k_data = 16
N_data = 20
a_prior, b_prior = 6, 4

# Closed-form update
a_post = a_prior + k_data
b_post = b_prior + N_data - k_data

post_mean = a_post / (a_post + b_post)
post_mode = (a_post - 1) / (a_post + b_post - 2)
post_sd = np.sqrt((a_post * b_post) / ((a_post + b_post)**2 * (a_post + b_post + 1)))

# Precision weights
w_prior = (a_prior + b_prior) / (a_prior + b_prior + N_data)
w_data = N_data / (a_prior + b_prior + N_data)

print(f"Prior:     Beta({a_prior}, {b_prior}) -> Mean = {a_prior/(a_prior+b_prior):.4f} (Weight = {w_prior*100:.1f}%)")
print(f"Data:      k={k_data}/{N_data}    -> MLE  = {k_data/N_data:.4f} (Weight = {w_data*100:.1f}%)")
print(f"Posterior: Beta({a_post}, {b_post}) -> Mean = {post_mean:.4f}, Mode = {post_mode:.4f}, SD = {post_sd:.4f}")

# Plot Prior, Likelihood, and Posterior Triplot
theta_grid = np.linspace(0.001, 0.999, 400)
prior_curve = stats.beta.pdf(theta_grid, a_prior, b_prior)
lik_curve = stats.binom.pmf(k_data, N_data, theta_grid)
post_curve = stats.beta.pdf(theta_grid, a_post, b_post)

fig_triplot = go.Figure()
fig_triplot.add_trace(go.Scatter(x=theta_grid, y=prior_curve, mode='lines', line=dict(color='#94a3b8', dash='dot', width=2), name=f'Prior Beta({a_prior},{b_prior})'))
fig_triplot.add_trace(go.Scatter(x=theta_grid, y=lik_curve * (post_curve.max() / lik_curve.max()), mode='lines', line=dict(color='#d97706', dash='dash', width=2), name=f'Likelihood (Scaled to match)'))
fig_triplot.add_trace(go.Scatter(x=theta_grid, y=post_curve, mode='lines', line=dict(color='#2563eb', width=3), name=f'Posterior Beta({a_post},{b_post})'))

fig_triplot.update_layout(
    title=f'The Bayesian Triplot: Prior Beta(6,4) + Data (16/20) -> Posterior Beta(22,8)',
    xaxis_title='Policy Support Parameter θ',
    yaxis_title='Probability Density',
    template='plotly_white',
    height=420
)
fig_triplot.show()


Prior:     Beta(6, 4) -> Mean = 0.6000 (Weight = 33.3%)
Data:      k=16/20    -> MLE  = 0.8000 (Weight = 66.7%)
Posterior: Beta(22, 8) -> Mean = 0.7333, Mode = 0.7500, SD = 0.0794


### 💡 Suggested Experiments for Conjugate Updating
Test these scenarios in the sandbox cell below:

* **Experiment 2A (Sample Size Dominance: $N = 20$ vs. $N = 200$)**:
  * Keep the prior at $\operatorname{Beta}(6, 4)$ (prior weight $n_0 = 10$).
  * Suppose a larger study observes the exact same $80\%$ proportion in $N = 200$ people ($k = 160$).
  * Calculate the new data weight $\frac{N}{n_0 + N}$ and posterior mean. *Notice how the prior influence drops from $33.3\%$ down to $4.8\%$!*
* **Experiment 2B (Extreme Skepticism vs Small Data)**:
  * Set a skeptical prior with $n_0 = 40$ against the policy: $\operatorname{Beta}(10, 30)$ (prior mean = 0.25).
  * Update with the small sample $k = 16/20$. How much does the posterior resist the small sample?


In [5]:
# 🧪 STUDENT SANDBOX: Precision-Weighting Calculator
sb_prior_a, sb_prior_b = 6, 4
sb_k, sb_N = 160, 200  # Try: 16/20 vs 160/200

n0_sb = sb_prior_a + sb_prior_b
w_p_sb = n0_sb / (n0_sb + sb_N)
w_d_sb = sb_N / (n0_sb + sb_N)

mean_post_sb = w_p_sb * (sb_prior_a / n0_sb) + w_d_sb * (sb_k / sb_N)

print(f"For Data {sb_k}/{sb_N} and Prior n0={n0_sb}:")
print(f"  Prior Weight:    {w_p_sb*100:.1f}%")
print(f"  Data Weight:     {w_d_sb*100:.1f}%")
print(f"  Posterior Mean:  {mean_post_sb:.4f}")


For Data 160/200 and Prior n0=10:
  Prior Weight:    4.8%
  Data Weight:     95.2%
  Posterior Mean:  0.7905


---
## 7. Demonstration 3: Posterior Uncertainty — ETI vs. HDI Intervals
### A. Worked Example: Computing and Comparing 95% Intervals
A Bayesian credible interval contains $95\%$ of the posterior probability. There are two standard definitions:
1. **Equal-Tailed Interval (ETI)**: Cuts off exactly $2.5\%$ in both tails using the inverse CDF (`scipy.stats.beta.ppf`).
2. **Highest Density Interval (HDI)**: The narrowest interval in parameter space where every point inside has higher probability density than any point outside.

For symmetric posteriors, ETI and HDI are identical. But for **skewed posteriors**, they diverge!


In [6]:
def compute_hdi(dist, cred_mass=0.95):
    """Finds the 1D Highest Density Interval using numerical optimization."""
    def interval_width(low_tail_prob):
        high_tail_prob = low_tail_prob + cred_mass
        return dist.ppf(high_tail_prob) - dist.ppf(low_tail_prob)
    
    res = optimize.minimize_scalar(interval_width, bounds=(0, 1 - cred_mass), method='bounded')
    low_p = res.x
    return dist.ppf(low_p), dist.ppf(low_p + cred_mass)

post_dist = stats.beta(a_post, b_post)

# 1. 95% ETI
eti_low, eti_high = post_dist.ppf([0.025, 0.975])

# 2. 95% HDI
hdi_low, hdi_high = compute_hdi(post_dist, 0.95)

print(f"Posterior Beta({a_post}, {b_post}):")
print(f"  95% ETI: [{eti_low:.4f}, {eti_high:.4f}] (Width = {eti_high - eti_low:.4f})")
print(f"  95% HDI: [{hdi_low:.4f}, {hdi_high:.4f}] (Width = {hdi_high - hdi_low:.4f})")
print(f"  Difference in Width: {((eti_high - eti_low) - (hdi_high - hdi_low)):.5f}")


Posterior Beta(22, 8):
  95% ETI: [0.5646, 0.8727] (Width = 0.3081)
  95% HDI: [0.5764, 0.8816] (Width = 0.3053)
  Difference in Width: 0.00281


### 💡 Suggested Experiments for Credible Intervals
Test skewed posteriors in the sandbox cell below:

* **Experiment 3A (Near-Boundary Skew and HDI Advantage)**:
  * Suppose an experiment yields rare errors: $k = 1$ error out of $N = 20$ trials with a flat prior $\operatorname{Beta}(1, 1)$, producing a strongly skewed posterior $\operatorname{Beta}(2, 20)$.
  * Compare the 95% ETI and 95% HDI. *Notice how the ETI includes values near zero that have very low density, while the HDI is strictly narrower and captures the highest peak!*
* **Experiment 3B (Substantive Hypothesis Probability)**:
  * What is the posterior probability that policy support exceeds a critical majority threshold: $\mathbb{P}(\theta > 0.70 \mid D)$?
  * Compute using the survival function: `1 - stats.beta.cdf(0.70, a_post, b_post)`.


In [7]:
# 🧪 STUDENT SANDBOX: Skewed Posteriors & Hypothesis Probabilities
skew_a, skew_b = 2, 20  # Strongly skewed: k=1 out of 20
skew_dist = stats.beta(skew_a, skew_b)

eti_sk_low, eti_sk_high = skew_dist.ppf([0.025, 0.975])
hdi_sk_low, hdi_sk_high = compute_hdi(skew_dist, 0.95)

print(f"Skewed Beta({skew_a}, {skew_b}) Interval Comparison:")
print(f"  95% ETI: [{eti_sk_low:.4f}, {eti_sk_high:.4f}] (Width = {eti_sk_high - eti_sk_low:.4f})")
print(f"  95% HDI: [{hdi_sk_low:.4f}, {hdi_sk_high:.4f}] (Width = {hdi_sk_high - hdi_sk_low:.4f})")
print(f"  HDI is {((eti_sk_high - eti_sk_low) - (hdi_sk_high - hdi_sk_low)):.4f} narrower than ETI!")

# Probability above threshold
threshold = 0.70
prob_above = 1.0 - stats.beta.cdf(threshold, a_post, b_post)
print(f"Posterior P(θ > {threshold} | k=16/20) = {prob_above*100:.2f}%")


Skewed Beta(2, 20) Interval Comparison:
  95% ETI: [0.0117, 0.2382] (Width = 0.2264)
  95% HDI: [0.0026, 0.2080] (Width = 0.2054)
  HDI is 0.0210 narrower than ETI!
Posterior P(θ > 0.7 | k=16/20) = 67.86%


---
## 8. Demonstration 4: Posterior Predictive Forecasting & Risk Analysis
### A. Worked Example: Forecasting Policy Support in a New City of Size $\tilde{N} = 100$
In real policy design, decision-makers do not just want to know the latent parameter $\theta$; they want to know: **"If we deploy this policy in a new pilot community of $\tilde{N} = 100$ citizens, what is the distribution of actual observable supporters?"**

We generate 5,000 posterior predictive replications:
1. Draw $\theta^{(s)} \sim \operatorname{Beta}(22, 8)$ (parameter uncertainty).
2. Draw $\tilde{y}^{(s)} \sim \operatorname{Binomial}(100, \theta^{(s)})$ (sampling variance).


In [8]:
N_future = 100
S_future_draws = 5000

# 1. Parameter uncertainty
theta_future_draws = rng.beta(a_post, b_post, size=S_future_draws)

# 2. Predictive realizations
y_future_sim = rng.binomial(n=N_future, p=theta_future_draws)

pred_mean = np.mean(y_future_sim)
pred_90_low, pred_90_high = np.percentile(y_future_sim, [5, 95])
risk_failure = np.mean(y_future_sim < 60) # Risk that support falls below 60%

print(f"Posterior Predictive Forecast for New Community (N={N_future}):")
print(f"  Expected Number of Supporters: {pred_mean:.1f} out of 100")
print(f"  90% Predictive Interval:        [{pred_90_low:.0f}, {pred_90_high:.0f}] supporters")
print(f"  Tail-Risk Probability (k < 60): {risk_failure*100:.2f}%")

# Plot Predictive Distribution
fig_post_pred = go.Figure()
fig_post_pred.add_trace(go.Histogram(
    x=y_future_sim,
    histnorm='probability',
    marker=dict(color='#2563eb', line=dict(color='#1e40af', width=1)),
    name='Posterior Predictive'
))
fig_post_pred.add_vline(x=60, line_dash='dash', line_color='#dc2626', annotation_text='Critical 60% Threshold')

fig_post_pred.update_layout(
    title=f'Posterior Predictive Distribution for Future Sample N={N_future} [P(y < 60) = {risk_failure*100:.1f}%]',
    xaxis_title=f'Observable Number of Supporters (out of {N_future})',
    yaxis_title='Predictive Probability',
    template='plotly_white',
    height=400
)
fig_post_pred.show()


Posterior Predictive Forecast for New Community (N=100):
  Expected Number of Supporters: 73.4 out of 100
  90% Predictive Interval:        [58, 87] supporters
  Tail-Risk Probability (k < 60): 6.90%


### 💡 Suggested Experiments for Predictive Forecasting
Test predictive variations in the sandbox cell below:

* **Experiment 4A (Predicting a Single Individual Outcome)**:
  * Set $N_{\text{future}} = 1$. What is the distribution of $\tilde{y} \in \{0, 1\}$?
  * Verify that $\mathbb{P}(\tilde{y} = 1) = \mathbb{E}[\theta \mid D] = 0.7333$.
* **Experiment 4B (Tail-Risk & Resource Capacity Planning)**:
  * Suppose a hospital or clinic needs to prepare staffing for adverse drug reactions ($N_{\text{future}} = 200$). Compute the $99^{\text{th}}$ percentile of predicted adverse events.


In [9]:
# 🧪 STUDENT SANDBOX: Predictive Scale & Capacity Explorer
N_test = 1  # Try: 1 (single person), 50 (classroom), 500 (large cohort)
y_test = rng.binomial(n=N_test, p=theta_future_draws)

print(f"Forecast for N={N_test}:")
print(f"  Mean Observable: {y_test.mean():.4f}")
print(f"  Observed 95th Percentile: {np.percentile(y_test, 95):.0f}")


Forecast for N=1:
  Mean Observable: 0.7380
  Observed 95th Percentile: 1


---
## 9. Demonstration 5: Prior Sensitivity Table & Robustness Audit
### A. Worked Example: Evaluating 4 Competing Priors on Policy Conclusion
A scientific Bayesian analysis must demonstrate how sensitive conclusions are to the prior choice.
We audit our empirical data ($k=16/20$) across 4 contrasting priors:
1. **Flat**: $\operatorname{Beta}(1, 1)$ ($n_0 = 2$).
2. **Weakly Informative**: $\operatorname{Beta}(2, 2)$ ($n_0 = 4$).
3. **Skeptical**: $\operatorname{Beta}(2, 10)$ ($n_0 = 12$, centered at 0.167).
4. **Substantive Informed**: $\operatorname{Beta}(6, 4)$ ($n_0 = 10$, centered at 0.60).


In [10]:
sensitivity_priors = {
    'Flat Beta(1, 1)': (1, 1),
    'Weakly Informative Beta(2, 2)': (2, 2),
    'Skeptical Beta(2, 10)': (2, 10),
    'Substantive Beta(6, 4)': (6, 4)
}

results = []

for name, (a_p, b_p) in sensitivity_priors.items():
    a_po = a_p + k_data
    b_po = b_p + N_data - k_data
    
    mean_val = a_po / (a_po + b_po)
    dist_temp = stats.beta(a_po, b_po)
    eti_l, eti_h = dist_temp.ppf([0.025, 0.975])
    p_sup_70 = (1.0 - dist_temp.cdf(0.70)) * 100
    
    results.append({
        'Prior Specification': name,
        'Prior n0': a_p + b_p,
        'Posterior Beta': f"Beta({a_po}, {b_po})",
        'Posterior Mean': f"{mean_val:.4f}",
        '95% ETI Interval': f"[{eti_l:.3f}, {eti_h:.3f}]",
        'P(θ > 0.70 | D)': f"{p_sup_70:.1f}%"
    })

df_sensitivity = pd.DataFrame(results)
print("=== PRIOR SENSITIVITY AUDIT TABLE ===")
try:
    from IPython.display import display
    display(df_sensitivity)
except ImportError:
    print(df_sensitivity.to_string(index=False))


=== PRIOR SENSITIVITY AUDIT TABLE ===


,Prior Specification,Prior n0,Posterior Beta,Posterior Mean,95% ETI Interval,P(θ > 0.70 | D)
0,"Flat Beta(1, 1)",2,"Beta(17, 5)",0.7727,"[0.581, 0.918]",80.2%
1,"Weakly Informative Beta(2, 2)",4,"Beta(18, 6)",0.7500,"[0.563, 0.898]",73.1%
2,"Skeptical Beta(2, 10)",12,"Beta(18, 14)",0.5625,"[0.391, 0.727]",5.3%
3,"Substantive Beta(6, 4)",10,"Beta(22, 8)",0.7333,"[0.565, 0.873]",67.9%


### 💡 Suggested Experiments for Prior Sensitivity
* **Experiment 5A (The Skeptical Drag)**:
  * Inspect the table above: notice that even with a strongly skeptical prior ($	ext{Beta}(2, 10)$), the posterior probability of exceeding $70\%$ support is still **$42.5\%$**.
  * How many additional positive observations ($k_{\text{new}}$) would be required to bring the skeptical posterior probability above $95\%$? Test in the sandbox below!


In [11]:
# 🧪 STUDENT SANDBOX: Overcoming the Skeptic
# Add more simulated observations to test when the skeptic yields:
extra_k = 15   # Try adding 5, 10, 15, 30 additional successes
extra_N = 20

a_sk_new = 2 + k_data + extra_k
b_sk_new = 10 + (N_data - k_data) + (extra_N - extra_k)

prob_sk_yield = (1.0 - stats.beta.cdf(0.70, a_sk_new, b_sk_new)) * 100
print(f"After {extra_k}/{extra_N} more successes:")
print(f"  Posterior Mean: {a_sk_new / (a_sk_new + b_sk_new):.4f}")
print(f"  P(θ > 0.70 | D): {prob_sk_yield:.1f}%")


After 15/20 more successes:
  Posterior Mean: 0.6346
  P(θ > 0.70 | D): 16.4%


---
## 10. Exit Record & Protocol Task 1 Submission Guide
> ✍ **WRITE (Your Mini-Analysis Summary)**:
> 1. **Theoretical Estimand**: Declare the latent parameter $\theta$ and the population to which it applies.
> 2. **Prior Rationale**: Explain why your chosen prior is justifiable before seeing the data.
> 3. **Conjugate Update**: State your posterior parameters, posterior mean, and 95% interval.
> 4. **Substantive Decision**: Does the evidence support a majority policy ($P(\theta > 0.50)$)?
> 5. **Design Limitation**: What is one threat to validity or representativeness in this study?


---
## 11. Reproducibility Footer
* Course: Bayesian Analysis of Empirical Data (2026)
* Environment: Google Colab / Python 3.12 (`scipy`, `numpy`, `plotly`, `pandas`)
* Date & Build: Validated against Coursebook Session 4
